# Convolutional Neural Networks for Image Classification

This notebook provides a complete workflow for training a CNN on the MNIST dataset, covering the transition from raw image data to a high-performance classifier.

---

## Learning Goals

* **CNN Fundamentals:** Understand how convolutional layers learn hierarchical spatial features (edges to shapes).
* **Deep Learning Workflow:** Implement data loading, preprocessing, and train/validation/test splits.
* **Regularization & Generalization:** Use early stopping and dropout to prevent overfitting.
* **Evaluation:** Move beyond accuracy to use confusion matrices and F1 scores for robust model assessment.
* **Transfer Learning:** Save learned feature representations for future reuse.

---

## The Pipeline

1. **Setup & Data:** Configure reproducibility, load MNIST, and apply normalization.
2. **Architecture:** Define a CNN with convolutional feature extraction and fully connected classification layers.
3. **Training:** Use mini-batch gradient descent with validation-based early stopping.
4. **Analysis:** Visualize learning curves, prediction errors via confusion matrices, and per-class performance.

---

## Core Concepts

### Feature Extraction

CNNs replace manual feature engineering by learning local spatial patterns—such as edges and curves—automatically.

### Model Selection

We use a **validation set** to monitor training, tune hyperparameters, and select the best model checkpoint to ensure the system generalizes to unseen data.

### Robust Metrics

Because accuracy can be misleading on imbalanced datasets, we utilize **F1 scores** and **classification reports** to identify systematic errors and class-specific performance.

---

## Expected Outcomes

By the end of this notebook, you will have a trained CNN capable of classifying handwritten digits and a set of **pretrained weights** ready for transfer learning applications.

# Imports and Environment Setup

This notebook utilizes:

* **Deep Learning:** PyTorch (with GPU acceleration) and Torchvision.
* **Evaluation:** Scikit-learn for comprehensive metrics (F1 score, confusion matrix).
* **Visualization:** Matplotlib for learning curves and dataset inspection.
* **Utilities:** Custom tools to ensure reproducibility and track training progress.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import multiprocessing
from PIL import Image
from tqdm import tqdm
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import MNIST

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)


# Reproducibility and Device Selection

To ensure consistent results, we fix random seeds for Python, NumPy, and PyTorch, mitigating variability from parameter initialization and data shuffling. The notebook also automatically detects and utilizes the most efficient available hardware: **CUDA** (NVIDIA), **MPS** (Apple Silicon), or **CPU**.

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


# Image Preprocessing

Neural networks require input in tensor format. We prepare images by resizing, converting to tensors, and normalizing pixel intensities:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

Using dataset-specific mean ($\mu$) and standard deviation ($\sigma$) stabilizes optimization and accelerates convergence during training.

In [ ]:
mnist_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.1307,),
        std=(0.3081,)
    )
])


# Loading the MNIST Dataset

MNIST is a standard benchmark containing 70,000 grayscale images of digits (0–9). We partition the training data into separate sets for:

* **Training:** Optimizing model parameters.
* **Validation:** Tuning hyperparameters and early stopping.
* **Testing:** Final, unbiased evaluation.

In [ ]:
# 1. Load the complete training dataset (60,000 images)
mnist_full_train = MNIST(
    root="data",
    #root="/leonardo/pub/userinternal/mcelori1/IntroductionToDeepLearning/Datasets/",
    train=True,
    download=False,
    transform=mnist_transforms
)

# 2. Partition into train (80% / 48,000) and validation (20% / 12,000)
# Passing a manual seed generator guarantees the exact same split across runs
generator = torch.Generator().manual_seed(42)
mnist_train_dataset, mnist_val_dataset = random_split(
    mnist_full_train, 
    [0.8, 0.2], 
    generator=generator
)

# 3. Load the test dataset (remains untouched, 10,000 images)
mnist_test_dataset = MNIST(
    root="data",
    #root="/leonardo/pub/userinternal/mcelori1/IntroductionToDeepLearning/Datasets/",
    train=False,
    download=False,
    transform=mnist_transforms
)

# DataLoaders

PyTorch `DataLoaders` provide efficient mini-batch iteration, which reduces memory usage, accelerates optimization, and stabilizes gradient estimates. We apply random shuffling to the training loader to prevent the model from learning artifacts tied to the order of the data.

In [ ]:
batch_size = 128

pin_memory = (device.type == "cuda")
num_workers = min(4, multiprocessing.cpu_count()) if (device.type == "cuda") else 0

persistent_workers = (num_workers > 0)

mnist_train_loader = DataLoader(
    mnist_train_dataset,
    batch_size=batch_size,
    shuffle=True, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)

mnist_val_loader = DataLoader(
    mnist_val_dataset,
    batch_size=batch_size,
    shuffle=False, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)

mnist_test_loader = DataLoader(
    mnist_test_dataset,
    batch_size=batch_size,
    shuffle=False, num_workers=num_workers,
    pin_memory=pin_memory, persistent_workers=persistent_workers
)


# Dataset Inspection

Visual inspection is essential to verify correct preprocessing, label accuracy, and class distribution. This step helps identify issues like corrupted samples, improper normalization, or significant class imbalance before training begins.

In [ ]:
def visualize_dataset(dataset, num_samples=8):
    mean = np.array([0.1307])
    std = np.array([0.3081])
    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 3))
    total_samples = len(dataset)
    random_indices = random.sample(range(total_samples), num_samples)
    if hasattr(dataset, "dataset"):
        classes = dataset.dataset.classes
    else:
        classes = dataset.classes
    for i, idx in enumerate(random_indices):
        image_tensor, label_idx = dataset[idx]
        img = image_tensor.squeeze().numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        class_letter = classes[label_idx]
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"Label: {class_letter}", fontsize=11, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_dataset(mnist_train_dataset, num_samples=8)


# Class Distribution Analysis

We compute the number of samples for each digit class to verify dataset balance.

Balanced datasets generally simplify optimization because the model receives comparable training signal across classes.

In [ ]:
class_names = [str(i) for i in range(10)]

train_indices = mnist_train_dataset.indices
train_labels = mnist_full_train.targets[train_indices].tolist()
class_counts = Counter(train_labels)

print("Number of images per class:\n")
for i, count in class_counts.items():
    print(f"{class_names[i]:30s}: {count}")


In [ ]:
counts = [class_counts[i] for i in range(len(class_names))]

plt.figure(figsize=(8,4))
plt.bar(range(len(class_names)), counts)
plt.xlabel("Class index")
plt.ylabel("Number of images")
plt.title("Dataset class distribution")
plt.show()


# Convolutional Neural Network Architecture

We define a compact CNN for digit classification, split into two functional blocks:

* **Feature Extractor:** Convolutional and pooling layers automatically learn hierarchical spatial features, ranging from simple edges and strokes to complex digit structures.
* **Classifier:** Fully connected layers map these extracted features to class probabilities.

This clear separation between feature extraction and classification is critical, as it allows us to reuse the backbone for transfer learning tasks.

```
Input
↓
Conv → ReLU → Pool
↓
Conv → ReLU → Pool
↓
Flatten
↓
Linear → ReLU → Dropout
↓
Linear
```

> Batch normalization standardizes intermediate activations during training using mini-batch statistics.
<br>This stabilizes gradient flow, allows larger learning rates, and often accelerates convergence.
<br>BatchNorm also maintains non-trainable running statistics (running mean, running variance).



## Visualizing the Spatial Transformations

To help your students intuitively track how spatial resolution shrinks while feature depth expands, here is a visual reference you can drop directly into your architectural summary markdown section:

| Pipeline Stage | Layer Type | Output Activation Shape ($C \times H \times W$) | Rationale |
| --- | --- | --- | --- |
| **Input** | Raw Preprocessed Image | $1 \times 28 \times 28$ | Single-channel grayscale MNIST sample. |
| **Stage 1 (Conv)** | `nn.Conv2d(1, 32, k=3, p=1)` | $32 \times 28 \times 28$ | Padding preserves spatial boundaries; channels expand to 32. |
| **Stage 1 (Pool)** | `nn.MaxPool2d(2)` | $32 \times 14 \times 14$ | $2\times2$ pooling downsamples spatial height and width by half. |
| **Stage 2 (Conv)** | `nn.Conv2d(32, 64, k=3, p=1)` | $64 \times 14 \times 14$ | Features deepen to collect more abstract structural shapes. |
| **Stage 2 (Pool)** | `nn.MaxPool2d(2)` | $64 \times 7 \times 7$ | Final spatial reduction. Features are now ready for flattening. |
| **Classifier** | `nn.Flatten()` | $3136$ vector elements | Total inputs passed directly to the first dense layer ($64 \times 7 \times 7$). |


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            # Regularizes dense classifier layers
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
model = SimpleCNN(num_classes=10).to(device)


To calculate the number of parameters per layer, we look at the weights and biases for each operation.

* **Conv2d:** `(kernel_size * kernel_size * in_channels * out_channels) + out_channels` (the bias).
* **BatchNorm2d:** `2 * num_features` (gamma and beta learnable parameters).
* **Linear:** `(in_features * out_features) + out_features` (bias).

Here is the parameter breakdown for your `SimpleCNN`:

| Layer | Type | Calculation | Parameters |
| --- | --- | --- | --- |
| `features.0` | Conv2d | (3 x 3 x 1 x 32) + 32 | 320 |
| `features.1` | BatchNorm2d | 2 x 32 | 64 |
| `features.4` | Conv2d | (3 x 3 x 32 x 64) + 64 | 18,496 |
| `features.5` | BatchNorm2d | 2 x 64 | 128 |
| `classifier.1` | Linear | (64 x 7 x 7 x 128) + 128 | 401,536 |
| `classifier.4` | Linear | (128 x 10) + 10 | 1,290 |
| **Total** |  |  | **421,834** |



In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


In [ ]:
tot, tra = count_parameters(model)
print(f"\nSimpleCNN | total params: {tot}\ttrainable params: {tra}")


# Early Stopping

As training progresses, models may begin to overfit, leading to improved training performance but declining generalization. 

Early stopping is a regularization technique that monitors validation metrics, saves the best-performing model checkpoint, and terminates training once improvement stalls. 

This ensures we capture the optimal model before overfitting occurs.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_acc = -float("inf")
        self.best_weights = None
        self.best_epoch = 0
        self.min_delta = min_delta 

    def step(self, model, val_loss, val_acc, epoch):

        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.best_epoch = epoch+1
            self.counter = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        self.counter += 1
        print(
            f"[EarlyStopping] Epoch {epoch+1 if epoch is not None else ''}: "
            f"No improvement → counter {self.counter}/{self.patience}"
        )
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)


# Evaluation Function

Our evaluation routine processes validation and test data while disabling gradients (`torch.no_grad()`). 

We set the model to `model.eval()` to ensure deterministic behavior, lower memory consumption, and correct application of layers like BatchNorm and Dropout.

In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


# Training on MNIST

The CNN is trained using supervised learning on the MNIST training split.

During optimization:
- convolutional filters learn reusable visual patterns,
- classifier layers learn class-specific decision boundaries,
- and validation performance is monitored after each epoch.

Optimization uses:
- Cross Entropy Loss for multi-class classification,
- AdamW for adaptive gradient optimization with decoupled weight decay,
- and Early Stopping for regularization.


> In PyTorch, **`nn.CrossEntropyLoss` includes both a Softmax activation and a Negative Log Likelihood loss.** 

You do **not** need to add an explicit Softmax layer to the end of your model. 

Internally, PyTorch performs the calculation with much higher numerical stability, preventing precision errors during training.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2
)

early_stopping = EarlyStopping(patience=10)

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("TRAINING CNN ON MNIST")

epochs = 100

for epoch in range(epochs):

    # Training
    model.train()

    total_train_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(mnist_train_loader):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total += y.size(0)
        total_train_loss += loss.item() * x.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == y).sum().item()
    train_loss = total_train_loss / total
    train_accuracy = correct / total

    # Validation
    val_loss, val_accuracy = evaluate(model, mnist_val_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # Epoch summary
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # Early stopping
    if early_stopping.step(model, val_loss, val_accuracy, epoch):
        print("\nEarly stopping triggered.")
        break


# Restore best model
early_stopping.restore(model)

print("\n================================================")
print("BEST MODEL SUMMARY")
print("================================================")
print(f"Best Epoch         : {early_stopping.best_epoch}")
print(f"Best Validation Acc: {early_stopping.best_acc:.4f}")
print()


# Training Curves

We track training and validation metrics across epochs to diagnose convergence, optimization stability, and overfitting. 

The highlighted minimum on the validation loss curve indicates the optimal model checkpoint retained by our early stopping strategy.


In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

plt.plot(epochs_range, train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

best_epoch = early_stopping.best_epoch
best_loss = early_stopping.best_loss
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Cross Entropy Loss", fontsize=11, labelpad=10)

plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
epochs_range = range(1, len(train_losses) + 1)
best_epoch = early_stopping.best_epoch
best_acc = early_stopping.best_acc

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

plt.plot(epochs_range, train_accuracies, label="Training Accuracy", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="#d95f02", linewidth=2.5, linestyle="--")

plt.scatter(best_epoch, best_acc, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

plt.title("Training vs Validation Accuracy", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Accuracy", fontsize=11, labelpad=10)

plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")
plt.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


# Final Test Evaluation

After training, we evaluate the best model checkpoint on the held-out test set. 

This provides an unbiased estimate of generalization, as the data was never used for optimization or model selection. 

We calculate:

* **Accuracy:** Overall correctness.
* **Macro/Weighted F1 Scores:** Balanced metrics that account for performance across all classes, protecting against biases inherent in simple accuracy.

> **Note**: this is practice for real-world workflows, this is practice for real-world workflows,

In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for x, y in mnist_test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions)
        all_labels.append(y)

all_predictions = torch.cat(all_predictions).cpu().numpy()
all_labels = torch.cat(all_labels).cpu().numpy()

accuracy = np.mean(all_predictions == all_labels)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n================================================")
print("FINAL TEST METRICS")
print("================================================")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1   : {f1_weighted:.4f}")


# Confusion Matrix

The confusion matrix visualizes:
- correct predictions,
- and systematic classification errors.

This analysis helps identify visually ambiguous digit pairs such as:
- 1 vs 7,
- 3 vs 8,
- and 6 vs 9.

Examining misclassification structure often provides more insight than aggregate metrics alone.

Many model errors are semantically understandable even when the model is highly accurate.

In [ ]:
class_names = [str(i) for i in range(10)]

cm = confusion_matrix(
    all_labels,
    all_predictions
)

fig, ax = plt.subplots(figsize=(6, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

ax.grid(False)

ax.set_xticklabels(class_names, rotation=0)

plt.title("MNIST Digits Confusion Matrix")
plt.show()


# Classification Report

The classification report summarizes:
- precision,
- recall,
- and F1 score

for each class individually.

This provides a more detailed view of model behavior and helps identify:
- difficult classes,
- asymmetric errors,
- and class-specific weaknesses.


In [ ]:
print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

# Saving Pretrained Features

After training, the learned convolutional feature extractor is saved to disk.

Only the feature extraction layers are stored because they will later be reused for transfer learning experiments.


### checkpoint_dir = Path("./checkpoints")
checkpoint_path = checkpoint_dir / "mnist_features.pth"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

#torch.save(model.state_dict(), "mnist_simplecnn_model.pth")
torch.save(model.features.state_dict(), checkpoint_path)
print(f"\nModel saved to: {checkpoint_path}")


# Conclusion

This notebook demonstrates the complete deep learning workflow for image classification. By building a CNN for MNIST, we explored:

* **Feature Learning:** CNNs learn hierarchical spatial patterns automatically.
* **Best Practices:** Data normalization, mini-batch optimization, and regularization stabilize training.
* **Generalization:** We use validation sets and early stopping to ensure models perform well on unseen data.
* **Evaluation:** Beyond simple accuracy, we utilize F1 scores and confusion matrices to diagnose class-specific performance.

By modularizing our architecture into a **feature extractor** and a **classifier**, we have established the foundation for transfer learning—a technique that enables us to reuse learned visual representations for new tasks.

This experiment illustrates how proper optimization and evaluation allow neural networks to extract powerful, hierarchical insights directly from raw data.